# Thinning H SAF ASCAT soil-moisture obs: what we tried, and what GEOSldas now does

H SAF ASCAT surface soil moisture (H121 CDR / H139 NRT) is posted on a **12.5 km Fibonacci grid**. Neighbouring obs
have strongly overlapping footprints and correlated errors, so we want to thin them before assimilation. This
notebook explains, with figures, how the thinning method was chosen:

| | namelist input | GEOSldas_GridComp | method | outcome |
|---|---|---|---|---|
| **original proposal** | `obs_param_nml(56:58)%thin_ratio` | commit `ca872c7`, `fibgrid_thinning.F90` (never pushed) | per file, bin the obs on a coarse Fibonacci lattice with `thin_ratio` × fewer points; keep the obs nearest each cell centre | right count, uniform in latitude, but **no minimum separation** (§3–5) |
| **implemented** | `obs_param_nml(56:58)%thin_dist_km` | commits `a499c55` + `5e220ff`, `thin_min_dist` in `clsm_ensupd_read_obs.F90` | per window, among the obs assigned to a model tile, visit obs in file order; keep an obs unless it is closer than `thin_dist_km` to an obs kept earlier | **guaranteed minimum separation** (§6) |

The story, one section each: (1) the grid's point numbering is not spatial; (2) so index thinning fails; (3) coarse
Fibonacci bins fix the count but not the separation; (4) why no keep-1-in-2 pattern can separate obs on a hexagonal
grid; (5) other equal-area bins behave the same; (6) greedy minimum-distance thinning does what we want;
(7) caveats; (8) what reaches the CF0360 tiles.

`5e220ff` changed only the order in the reader: obs are matched to tiles **first**, and only obs with a tile are
thinned, so obs outside the domain or without a tile cannot suppress obs that can be assimilated. `thin_min_dist`
itself is unchanged. §6–7 thin all QC'd obs to show the method on its own; §8 follows the reader.

**To try other namelist values**, change `THIN_RATIO` and/or `THIN_DIST_KM` in the settings cell (§0) and re-run all.

Runs locally in the `regrid` env in about 2–3 minutes. Data: local H121 files (Jan 2020), the CF0360 tilecoord file from
`GEOSldas_diagnostics/test_data`.

The Python versions of both thinning methods were checked against the compiled Fortran (`thin_min_dist` from
`a499c55`, `fibgrid_thinning.F90` from `ca872c7`) in a standalone test, Oct 2026: identical kept obs on the same inputs.

## 0. Setup

### Settings: the namelist inputs

- `THIN_RATIO`: `obs_param_nml%thin_ratio` of the original proposal (ratio of obs grid points to coarse cells;
  2 = "every other obs").
- `THIN_DIST_KM`: `obs_param_nml%thin_dist_km` as implemented (minimum distance between assimilated obs).

Both are added to the comparison lists automatically, so every table and multi-panel figure includes them.

In [ ]:
THIN_RATIO   = 2.0       # obs_param_nml(56:58)%thin_ratio    (original proposal, ca872c7; 0 = off)
THIN_DIST_KM = 20.0      # obs_param_nml(56:58)%thin_dist_km  (implemented, a499c55 + 5e220ff; 0 = off)

THIN_RATIOS_COMPARE = sorted({2.0, 3.0, 4.0, THIN_RATIO} - {0.0})
THIN_DISTS_COMPARE  = sorted({15.0, 20.0, 25.0, THIN_DIST_KM} - {0.0})

# assimilation windows (GEOSldas date_time = window centre, window = (date_time - 1.5 h, date_time + 1.5 h])
CYCLE    = '2020-01-15 01:30'     # main window, Metop-B: obs from 00:00 to 03:00 UTC
SAT      = 'b'
CYCLE_BC = '2020-01-03 01:30'     # window for the Metop-B vs Metop-C caveat (both satellites have data)

# small patches for maps (lat0, lat1, lon0, lon1)
PATCH_SWATH = (-12.0, -10.0, -48.0, -46.0)     # inside a Metop-B swath in the main window (Brazil)

In [ ]:
import glob
import re
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import netCDF4 as nc
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from scipy import sparse
from scipy.spatial import cKDTree

%matplotlib inline
T_START = time.time()

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / '.git').exists():
            return p
    return start

REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / 'common' / 'python' / 'io'))
from read_GEOSldas import read_tilecoord

H121_DIR  = Path('/Users/amfox/Desktop/ASCAT_SSM_CDR/H121')         # metop_{a,b,c}/YYYY/MM/*.nc
H1XX_GLOB = '/Users/amfox/Desktop/ASCAT_SSM_CDR/H1*/**/*.nc'      # H121 + H139 (not H29)
TILECOORD = Path('/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/land_sweeper/LS_OLv8_M36/output/'
                 'SMAP_EASEv2_M36_GLOBAL/rc_out/M21C_test_CF0360.ldas_tilecoord.bin')

FIG_DIR = REPO_ROOT / 'projects' / 'M21C_land' / 'figures' / 'obs_thinning_explained'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# consistent colours: grey = all obs, blue = kept, red = too-close pairs, black x = cell centres
C_ALL, C_KEEP, C_BAD = '0.72', 'tab:blue', 'tab:red'

### Helper functions

Lattice and geometry helpers, the thinning methods, and the H SAF reader. The names map to the Fortran as follows:

| Python | Fortran |
|---|---|
| `n_thin(thin_ratio)` | `N_fib_thin = nint(real(N_fib_obs_grid)/thin_ratio)` in the `ca872c7` reader |
| `thin_fib_bins` | `fib_thin` in `fibgrid_thinning.F90` (KD-tree instead of the bucket search; same exact nearest point) |
| `thin_min_dist` | `thin_min_dist` in `clsm_ensupd_read_obs.F90` (KD-tree pairs instead of the bucket grid; same rule) |
| `read_hsaf_window` | file selection + QC of `read_obs_sm_ASCAT_HSAF` |

In [ ]:
PHI = (1 + 5 ** 0.5) / 2
R_EARTH_KM = 6371.0
E2_WGS84 = 0.00669437999014
N_FINE = 1_650_000                      # 12.5 km product (reader parameter N_fib_obs_grid)


def fib_latlon(n):
    """Spherical Fibonacci lattice (TU Wien convention). Returns i (=-n..n), lat, lon [deg], lon in [-180, 180)."""
    i = np.arange(-n, n + 1, dtype=np.int64)
    lat = np.degrees(np.arcsin(2.0 * i / (2.0 * n + 1.0)))
    lon = ((i % PHI) * 360.0 / PHI + 180.0) % 360.0 - 180.0
    return i, lat, lon


def location_id_to_i(loc, n=N_FINE):
    """H SAF location_id -> lattice index i (location_id = i for i >= 0, i + 2n+1 for i < 0)."""
    return np.where(loc <= n, loc, loc - (2 * n + 1))


def spherical_to_geodetic(lat_deg):
    return np.degrees(np.arctan(np.tan(np.radians(lat_deg)) / (1 - E2_WGS84)))


def to_xyz(lat, lon):
    la, lo = np.radians(np.asarray(lat, float)), np.radians(np.asarray(lon, float))
    return np.column_stack((np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)))


def chord_to_km(c):
    return 2 * np.arcsin(np.asarray(c) / 2) * R_EARTH_KM


def nn_spacing_km(lat, lon):
    """Great-circle distance from each point to its nearest other point."""
    p = to_xyz(lat, lon)
    d, _ = cKDTree(p).query(p, 2)
    return chord_to_km(d[:, 1])


def spacing_by_lat_band(lat, lon, bands=(0, 15, 30, 45, 60, 75, 90), d=None):
    d = nn_spacing_km(lat, lon) if d is None else d
    a = np.abs(lat)
    return {f"{lo}-{hi}": d[(a >= lo) & (a < hi)].mean()
            for lo, hi in zip(bands[:-1], bands[1:]) if np.any((a >= lo) & (a < hi))}


def sqrt_area_per_ob_km(n_obs):
    return np.sqrt(4 * np.pi * R_EARTH_KM ** 2 / n_obs)


def close_pairs(lat, lon, r_km):
    """index pairs (i < j) closer than r_km (strictly)"""
    p = to_xyz(lat, lon)
    chord = 2 * np.sin(r_km / R_EARTH_KM / 2)
    pr = cKDTree(p).query_pairs(chord, output_type='ndarray')
    if pr.size == 0:
        return pr.reshape(0, 2)
    return pr[((p[pr[:, 0]] - p[pr[:, 1]]) ** 2).sum(1) < chord ** 2]


# ---------------------------------------------------------------- binning (original proposal)

def select_one_per_cell(cell, score):
    """Index of the min-score member of each cell (ties -> lowest index)."""
    order = np.lexsort((np.arange(cell.size), score, cell))
    first = np.r_[True, cell[order][1:] != cell[order][:-1]]
    return np.sort(order[first])


_COARSE_TREES = {}

def coarse_cells(lat, lon, n_coarse):
    """nearest coarse lattice point (cell = i + n_coarse) and chord distance to it"""
    if n_coarse not in _COARSE_TREES:
        _, clat, clon = fib_latlon(n_coarse)
        _COARSE_TREES[n_coarse] = cKDTree(to_xyz(clat, clon))
    dist, cell = _COARSE_TREES[n_coarse].query(to_xyz(lat, lon))
    return cell, dist


def thin_fib_bins(lat, lon, n_coarse):
    """Binning on a coarse Fibonacci lattice; keep the obs closest to each cell centre. Returns kept indices."""
    cell, dist = coarse_cells(lat, lon, n_coarse)
    return select_one_per_cell(cell, dist)


def n_thin(thin_ratio):
    """N_fib_thin = nint( real(N_fib_obs_grid)/thin_ratio ) as in the ca872c7 reader (single precision)"""
    x = np.float32(N_FINE) / np.float32(thin_ratio)
    return int(np.floor(x + np.float32(0.5)))


def thin_reduced_latlon(lat, lon, h_km):
    """Binning on a reduced lat/lon grid (rows h_km tall, ~h_km-wide columns); keep obs closest to cell centre."""
    h = np.degrees(h_km / R_EARTH_KM); nrow = int(np.ceil(180 / h))
    row = np.minimum(((lat + 90) / h).astype(int), nrow - 1)
    latc = -90 + (np.arange(nrow) + 0.5) * h
    ncol = np.maximum(1, np.rint(360 * np.cos(np.radians(latc)) / h).astype(int))
    off = np.r_[0, np.cumsum(ncol)[:-1]]
    w = 360 / ncol[row]
    col = np.minimum((((lon + 180) % 360) / w).astype(int), ncol[row] - 1)
    d = to_xyz(lat, lon) - to_xyz(latc[row], -180 + (col + 0.5) * w)
    return select_one_per_cell(off[row] + col, (d * d).sum(1))


# ---------------------------------------------------------------- greedy minimum distance (implemented)

def thin_min_dist_reference(lat, lon, r_km):
    """Greedy minimum-distance thinning, literal version (same rule as GEOSldas thin_min_dist in clsm_ensupd_read_obs.F90):
    visit obs in the given order; keep an obs unless it is closer than r_km to an obs kept earlier.
    Returns a boolean keep mask. Pure Python; use for checks on <~300k obs."""
    p = to_xyz(lat, lon)
    chord = 2 * np.sin(r_km / R_EARTH_KM / 2)
    nbrs = cKDTree(p).query_ball_point(p, chord)
    keep = np.zeros(len(p), bool)
    for k in range(len(p)):
        keep[k] = not any(keep[j] and j != k and ((p[j] - p[k]) ** 2).sum() < chord ** 2 for j in nbrs[k])
    return keep


def thin_min_dist(lat, lon, r_km):
    """Same rule and result as thin_min_dist_reference, faster: for each obs, the list of EARLIER obs closer than
    r_km is built once (KD-tree pairs, sparse matrix); the greedy pass then only checks those."""
    n = len(lat)
    keep = np.zeros(n, bool)
    if n == 0:
        return keep
    pr = close_pairs(lat, lon, r_km)
    earlier = sparse.csr_matrix((np.ones(len(pr), bool), (pr.max(1), pr.min(1))), shape=(n, n))
    ip, ix = earlier.indptr, earlier.indices
    for k in range(n):
        keep[k] = not keep[ix[ip[k]:ip[k + 1]]].any()
    return keep


def min_sep_km(lat, lon):
    return nn_spacing_km(lat, lon).min() if len(lat) > 1 else np.inf


# ---------------------------------------------------------------- H SAF reader

QC_VARS = ['location_id', 'latitude', 'longitude', 'time', 'surface_soil_moisture', 'surface_flag',
           'processing_flag', 'backscatter40_flag', 'wetland_fraction', 'topographic_complexity',
           'subsurface_scattering_probability', 'surface_soil_moisture_sensitivity']


def read_hsaf(path, t_low=-np.inf, t_up=np.inf):
    """QC'd obs of one H SAF file, in file order (QC as read_obs_sm_ASCAT_HSAF).
    Raw packed values are read (no auto mask/scale), exactly like nf90_get_var in the reader, and lat/lon are
    formed in single precision, real(lat_raw)*real(1e-6), because that is what the thinning routines receive.
    t_low/t_up: obs time window (days since 1970), half-open (t_low, t_up]."""
    with nc.Dataset(path) as d:
        d.set_auto_maskandscale(False)
        v = {k: d[k][:] for k in QC_VARS}
    ssm = v['surface_soil_moisture']
    sens = v['surface_soil_moisture_sensitivity'].astype(np.float32) * np.float32(1e-7)     # [dB]
    wet = v['wetland_fraction'].astype(np.int8).astype(int)
    topo = v['topographic_complexity'].astype(np.int8).astype(int)
    subs = v['subsurface_scattering_probability'].astype(np.int8).astype(int)
    in_win = (ssm >= 0) & (ssm <= 10000) & (v['time'] > t_low) & (v['time'] <= t_up)
    qc = (in_win
          & ((v['surface_flag'].astype(np.uint8) & 1) == 0)          # open water; fill 255 -> rejected
          & ((v['processing_flag'].astype(np.uint8) & 3) == 0)
          & ((v['backscatter40_flag'] & 4) == 0)                     # noise out of limits
          & (wet >= 0) & (wet < 10)                                  # fill (-128) -> rejected
          & (topo >= 0) & (topo < 10)
          & (subs < 5)                                               # fill (-128) -> accepted
          & (sens > 1.0))
    lat = v['latitude'].astype(np.float32) * np.float32(1e-6)
    lon = v['longitude'].astype(np.float32) * np.float32(1e-6)
    return dict(lat=lat[qc], lon=lon[qc], time=v['time'][qc], ssm=ssm[qc].astype(np.float32) * np.float32(0.01),
                sens=sens[qc], location_id=v['location_id'][qc], n_file=ssm.size, n_window=int(in_win.sum()))


def read_hsaf_window(sat, date_time, dtstep=10800):
    """QC'd obs of all files of one assimilation window, in file order, like read_obs_sm_ASCAT_HSAF:
    window (date_time - dtstep/2, date_time + dtstep/2]; files whose sensing start is in (window start - 1 h, window end]."""
    dt = datetime.strptime(date_time, '%Y-%m-%d %H:%M') if isinstance(date_time, str) else date_time
    low, up = dt - timedelta(seconds=dtstep // 2), dt + timedelta(seconds=dtstep // 2)
    low_fname = low - timedelta(seconds=3600)
    epoch = datetime(1970, 1, 1)
    t_low, t_up = ((t - epoch).total_seconds() / 86400 for t in (low, up))
    files = []
    for f in glob.glob(str(H121_DIR / f'metop_{sat}' / '*' / '*' / '*.nc')):
        start = datetime.strptime(re.search(r'_(\d{14})_\d{14}____\.nc$', f).group(1), '%Y%m%d%H%M%S')
        if low_fname < start <= up:
            files.append((start, f))
    parts, rows = [], []
    for k, (start, f) in enumerate(sorted(files)):
        o = read_hsaf(f, t_low, t_up)
        o['file'] = np.full(o['lat'].size, k)
        parts.append(o)
        rows.append({'file sensing start': f'{start:%Y-%m-%d %H:%M}', 'obs in file': o['n_file'],
                     'in window': o['n_window'], 'passed QC': o['lat'].size})
    keys = ['lat', 'lon', 'time', 'ssm', 'sens', 'location_id', 'file']
    W = {k: np.concatenate([p[k] for p in parts]) if parts else np.array([]) for k in keys}
    W['table'] = pd.DataFrame(rows)
    W['label'] = f'Metop-{sat.upper()} window ({low:%Y-%m-%d %H:%M}, {up:%H:%M}]'
    return W


def as64(*a):
    return [np.asarray(x, np.float64) for x in a]


def patch_mask(lat, lon, patch):
    la0, la1, lo0, lo1 = patch
    return (lat >= la0) & (lat < la1) & (lon >= lo0) & (lon < lo1)


def set_patch(ax, patch):
    la0, la1, lo0, lo1 = patch
    ax.set_xlim(lo0, lo1); ax.set_ylim(la0, la1)
    ax.set_aspect(1 / np.cos(np.radians(0.5 * (la0 + la1))))
    ax.set_xlabel('lon [deg]'); ax.set_ylabel('lat [deg]')


def draw_pairs(ax, lat, lon, pairs, **kw):
    seg = np.stack([np.c_[lon[pairs[:, 0]], lat[pairs[:, 0]]], np.c_[lon[pairs[:, 1]], lat[pairs[:, 1]]]], axis=1)
    ax.add_collection(LineCollection(seg, **kw))


def fmt(v, nd=1):
    return f'{v:.{nd}f}'

## 1. The H SAF Fibonacci grid

A Fibonacci lattice of half-size $N$ has $2N+1$ points $i = -N \dots N$, with $\mathrm{lat}_i = \arcsin(2i/(2N+1))$ and
$\mathrm{lon}_i = 360^\circ\,(i \bmod \varphi)/\varphi$. Every point represents the same area. Consecutive $i$ are
~137.5° apart in longitude, so **index order is not spatial order**. The 12.5 km product has $N$ = 1,650,000.

In [ ]:
I_FULL, LAT_SPH, LON_FULL = fib_latlon(N_FINE)
LAT_GD = spherical_to_geodetic(LAT_SPH)                 # file latitudes are geodetic (1.2)
NN_FULL = nn_spacing_km(LAT_SPH, LON_FULL)
TREE_FULL = cKDTree(to_xyz(LAT_SPH, LON_FULL))
print(f'N = {N_FINE:,}: {I_FULL.size:,} points')
print(f'mean nearest-neighbour spacing = {NN_FULL.mean():.2f} km;  sqrt(4 pi R^2/(2N+1)) = {sqrt_area_per_ob_km(I_FULL.size):.2f} km')

### 1.2 How the files number the points, and which latitude they use

Three ways to place a file's `location_id` on the lattice: (a) the `fibgrid`-docs convention `gpi = i + N`;
(b) the correct mapping `i = location_id` (≤ N) or `location_id − (2N+1)`, spherical latitude; (c) the correct mapping
plus the geodetic conversion $\tan\phi = \tan\psi/(1-e^2)$.

In [ ]:
TEST_FILE = sorted(glob.glob(H1XX_GLOB, recursive=True))[0]
with nc.Dataset(TEST_FILE) as d:
    loc_t = np.asarray(d['location_id'][:], np.int64)
    flat_t = np.asarray(d['latitude'][:], float)          # auto-scaled by netCDF4
    flon_t = np.asarray(d['longitude'][:], float)

def dlon(a, b):
    return np.abs((a - b + 180) % 360 - 180)

i_a = loc_t - N_FINE                                       # (a) naive
i_b = location_id_to_i(loc_t)                              # (b), (c)
rows = {}
for name, ii, geo in [('(a) gpi = i + N', i_a, False), ('(b) correct mapping, spherical lat', i_b, False),
                      ('(c) correct mapping, geodetic lat', i_b, True)]:
    la, lo = LAT_SPH[ii + N_FINE], LON_FULL[ii + N_FINE]
    la = spherical_to_geodetic(la) if geo else la
    rows[name] = {'max |dlat| [deg]': np.abs(la - flat_t).max(), 'max |dlon| [deg]': dlon(lo, flon_t).max()}
MAP_CHECK = pd.DataFrame(rows).T
print(f'{Path(TEST_FILE).name}: {loc_t.size:,} obs')
print(MAP_CHECK.to_string(float_format=lambda v: f'{v:.2e}'))

### 1.3 Neighbours have very different indices

Small patches at the equator and at 60°N, each lattice point labelled with its `location_id`.

In [ ]:
LOC_FULL = np.where(I_FULL >= 0, I_FULL, I_FULL + 2 * N_FINE + 1)
fig, axs = plt.subplots(1, 2, figsize=(16, 7.5), constrained_layout=True)
for ax, (la0, lo0) in zip(axs, [(0.0, 20.0), (60.0, 20.0)]):
    w = 0.45
    patch = (la0, la0 + w, lo0, lo0 + w / np.cos(np.radians(la0)))
    m = patch_mask(LAT_GD, LON_FULL, patch)
    ax.plot(LON_FULL[m], LAT_GD[m], 'o', color=C_KEEP, ms=5)
    for x, y, g in zip(LON_FULL[m], LAT_GD[m], LOC_FULL[m]):
        ax.text(x, y + 0.008, f'{g}', fontsize=7, ha='center')
    set_patch(ax, patch)
    ax.set_title(f'{la0:.0f}°N: {m.sum()} points labelled with location_id (~{w * 111:.0f} km patch)')
fig.savefig(FIG_DIR / 'gpi_labels.png', dpi=150)
plt.show()

### 1.4 Neighbour index offsets are latitude-dependent Fibonacci numbers

In [ ]:
rows = []
for lat0 in [0, 30, 60, 80]:
    j0 = int(np.argmin(np.abs(LAT_SPH - lat0)))
    d, nb = TREE_FULL.query(to_xyz(LAT_SPH[j0], LON_FULL[j0]), 7)
    rows.append({'lat': lat0, '|Δi| of the 6 nearest neighbours': sorted(np.abs(nb[0][1:] - j0).tolist()),
                 'distances [km]': np.round(np.sort(chord_to_km(d[0][1:])), 1).tolist()})
print(pd.DataFrame(rows).to_string(index=False))

**Takeaway:** gpi order is not spatial order. Locally the grid is close to hexagonal: about 6 neighbours at nearly
equal distance, in directions that change with latitude.

## 2. Why index-based thinning fails

Keep `i % 2 == 0` or `i % 4 == 0` on the full lattice and look at the mean nearest-neighbour spacing per latitude band.

In [ ]:
IDX_SETS = {'full grid': np.ones(I_FULL.size, bool), 'i % 2 == 0': I_FULL % 2 == 0, 'i % 4 == 0': I_FULL % 4 == 0}
rows = {}
for name, m in IDX_SETS.items():
    d = NN_FULL if name == 'full grid' else nn_spacing_km(LAT_SPH[m], LON_FULL[m])
    rows[name] = spacing_by_lat_band(LAT_SPH[m], LON_FULL[m], d=d)
IDX_TABLE = pd.DataFrame(rows).T
print('mean nearest-neighbour spacing [km] per |lat| band')
print(IDX_TABLE.to_string(float_format=fmt))

fig, axs = plt.subplots(1, 2, figsize=(14, 6.5), constrained_layout=True)
for ax, la0 in zip(axs, [0.0, 60.0]):
    patch = (la0, la0 + 1.0, 20.0, 20.0 + 1.0 / np.cos(np.radians(la0)))
    b = patch_mask(LAT_SPH, LON_FULL, patch)
    k = IDX_SETS['i % 4 == 0']
    ax.plot(LON_FULL[b & ~k], LAT_SPH[b & ~k], 'o', color=C_ALL, ms=3)
    ax.plot(LON_FULL[b & k], LAT_SPH[b & k], 'o', color=C_KEEP, ms=6)
    set_patch(ax, patch)
    ax.set_title(f'i % 4 == 0 at {la0:.0f}°N: mean NN {IDX_TABLE.iloc[2, 0 if la0 == 0 else 4]:.1f} km')
fig.savefig(FIG_DIR / 'index_thinning_patches.png', dpi=150)
plt.show()

**Takeaway:** the result depends strongly on latitude. `% 4` drops 75% of the tropical obs and leaves their spacing
unchanged (they still form lines at the original spacing). Don't use it.

## 3. The original proposal: binning on a coarse Fibonacci lattice (`thin_ratio`)

Bin obs on a coarse Fibonacci lattice with `thin_ratio` times fewer points and keep the obs nearest each cell centre.
Here on the full 12.5 km lattice at geodetic latitudes, like the files (obs geodetic, coarse lattice spherical).

In [ ]:
def stats_row(lat, lon, n_total=None):
    d = nn_spacing_km(lat, lon)
    return {'kept': lat.size / (n_total or lat.size) * 100, 'sqrt(area/ob) [km]': sqrt_area_per_ob_km(lat.size),
            'NN mean [km]': d.mean(), 'NN p05 [km]': np.percentile(d, 5), 'NN min [km]': d.min()}, d

BIN_KEEP, BIN_NN = {}, {}
rows = {'1 (all)': stats_row(LAT_GD, LON_FULL)[0]}
for r in THIN_RATIOS_COMPARE:
    keep = thin_fib_bins(LAT_GD, LON_FULL, int(round(N_FINE / r)))
    BIN_KEEP[r] = keep
    rows[f'{r:g}'], BIN_NN[r] = stats_row(LAT_GD[keep], LON_FULL[keep], LAT_GD.size)
BIN_TABLE = pd.DataFrame(rows).T
BIN_TABLE.index.name = 'thin_ratio'
print(BIN_TABLE.to_string(float_format=fmt))

r0 = THIN_RATIO
band = spacing_by_lat_band(LAT_GD[BIN_KEEP[r0]], LON_FULL[BIN_KEEP[r0]], d=BIN_NN[r0])
print(f'\nthin_ratio = {r0:g}: mean NN spacing per |lat| band [km]:', {k: round(v, 1) for k, v in band.items()})

### 3.3 Missing obs are replaced by the next-best obs in the cell

Remove 30% of the points at random (seed 1) and thin the rest. A cell whose "ideal" obs is missing keeps another
valid obs instead.

In [ ]:
VALID = np.random.default_rng(1).random(I_FULL.size) < 0.7
lat_v, lon_v = LAT_GD[VALID], LON_FULL[VALID]
nc_ = int(round(N_FINE / THIN_RATIO))
keep_v = thin_fib_bins(lat_v, lon_v, nc_)
cell_full, _ = coarse_cells(LAT_GD, LON_FULL, nc_)
ideal = np.zeros(I_FULL.size, bool); ideal[BIN_KEEP[THIN_RATIO]] = True
ideal_missing_cells = np.unique(cell_full[ideal & ~VALID])
kept_cells = np.unique(cell_full[VALID][keep_v])
print(f'thin_ratio = {THIN_RATIO:g}, 70% valid: kept {keep_v.size:,} of {VALID.sum():,} valid obs ({keep_v.size / VALID.sum():.1%})')
print(f'cells whose ideal obs is missing: {ideal_missing_cells.size:,}; of those, '
      f'{np.isin(ideal_missing_cells, kept_cells).mean():.1%} still keep a (replacement) obs')

### 3.4 Kept obs in a patch, and the pairs that are still close

All obs grey, coarse cell centres ×, kept obs blue; red lines join **kept pairs closer than 13 km**, i.e. pairs
straddling a cell boundary that are still neighbouring 12.5 km grid points.

In [ ]:
PATCH_LAT = (34.0, 35.0, -92.0, -92.0 + 1.0 / np.cos(np.radians(34.5)))
b_all = patch_mask(LAT_GD, LON_FULL, PATCH_LAT)
fig, axs = plt.subplots(1, len(THIN_RATIOS_COMPARE), figsize=(6.2 * len(THIN_RATIOS_COMPARE), 6.4),
                        constrained_layout=True)
for ax, r in zip(np.atleast_1d(axs), THIN_RATIOS_COMPARE):
    nc_r = int(round(N_FINE / r))
    k = np.zeros(I_FULL.size, bool); k[BIN_KEEP[r]] = True
    ax.plot(LON_FULL[b_all], LAT_GD[b_all], 'o', color=C_ALL, ms=4)
    _, clat, clon = fib_latlon(nc_r)
    mc = patch_mask(clat, clon, PATCH_LAT)
    ax.plot(clon[mc], clat[mc], 'kx', ms=6)
    la, lo = LAT_GD[b_all & k], LON_FULL[b_all & k]
    pr = close_pairs(la, lo, 13.0)
    draw_pairs(ax, la, lo, pr, colors=C_BAD, linewidths=2, zorder=3)
    ax.plot(lo, la, 'o', color=C_KEEP, ms=6, zorder=4)
    set_patch(ax, PATCH_LAT)
    ax.set_title(f'thin_ratio = {r:g} (cells ~{sqrt_area_per_ob_km(2 * nc_r + 1):.1f} km)\n'
                 f'{la.size} kept, {len(pr)} kept pairs < 13 km')
fig.savefig(FIG_DIR / 'fib_binning_patch.png', dpi=150)
plt.show()

**Takeaway:** the count is right and uniform across latitude, but at 1-in-2 the nearest-neighbour distance barely
grows (12.1 → 12.8 km): many kept obs are still direct neighbours on the 12.5 km grid.

## 4. Why keeping "every other" obs cannot separate them

Take an ideal hexagonal lattice with spacing $a = 1$ and basis $a_1 = (1, 0)$, $a_2 = (1/2, \sqrt3/2)$. Its six nearest
neighbours lie along three directions: $a_1$, $a_2$, $a_2 - a_1$.

Every regular keep-1-in-2 pattern is a sub-lattice of index 2. There are exactly three, the kernels of the three
non-zero maps $\mathbb{Z}^2 \to \mathbb{Z}_2$: $L_1 = \{m\,a_1 + n\,a_2: m \text{ even}\}$, $L_2 = \{n \text{ even}\}$,
$L_3 = \{m + n \text{ even}\}$. Each contains one nearest-neighbour direction ($L_1 \ni a_2$, $L_2 \ni a_1$,
$L_3 \ni a_2 - a_1$), so its minimum spacing is still $a$: rows of points at spacing $a$, rows $\sqrt3\,a$ apart.
Keep-1-in-3 (the $\sqrt3$ sub-lattice, $m \equiv n \pmod 3$) and keep-1-in-4 (the 2× lattice) do separate the points.

In [ ]:
mm, nn_ = np.meshgrid(np.arange(-12, 13), np.arange(-12, 13))
mm, nn_ = mm.ravel(), nn_.ravel()
HX = np.c_[mm + 0.5 * nn_, np.sqrt(3) / 2 * nn_]
SUBS = {'L1: m even (1/2)': mm % 2 == 0, 'L2: n even (1/2)': nn_ % 2 == 0, 'L3: m+n even (1/2)': (mm + nn_) % 2 == 0,
        '√3 lattice: m≡n mod 3 (1/3)': (mm - nn_) % 3 == 0, '2× lattice: m, n even (1/4)': (mm % 2 == 0) & (nn_ % 2 == 0)}

def min_spacing_2d(pts):
    d, _ = cKDTree(pts).query(pts, 2)
    return d[:, 1].min()

fig, axs = plt.subplots(1, 5, figsize=(22, 4.8), constrained_layout=True)
rows = []
for ax, (name, m) in zip(axs, SUBS.items()):
    s = min_spacing_2d(HX[m])
    rows.append({'pattern': name, 'kept fraction': m.mean(), 'min spacing [a]': s})
    ax.plot(HX[~m, 0], HX[~m, 1], 'o', color=C_ALL, ms=4)
    ax.plot(HX[m, 0], HX[m, 1], 'o', color=C_KEEP, ms=7)
    # draw the nearest-neighbour links that survive
    pr = cKDTree(HX[m]).query_pairs(s + 1e-9, output_type='ndarray')
    draw_pairs(ax, HX[m][:, 1], HX[m][:, 0], pr, colors=C_BAD if s < 1.01 else 'k', linewidths=1.2)
    ax.set_xlim(-4, 4); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
    ax.set_title(f'{name}\nmin spacing {s:.3f} a', fontsize=11)
fig.savefig(FIG_DIR / 'hex_sublattices.png', dpi=150)
plt.show()

HEX_TABLE = pd.DataFrame(rows)
real = {0.5: BIN_TABLE.loc['2', 'NN min [km]'] if '2' in BIN_TABLE.index else np.nan,
        1 / 3: BIN_TABLE.loc['3', 'NN min [km]'] if '3' in BIN_TABLE.index else np.nan,
        0.25: BIN_TABLE.loc['4', 'NN min [km]'] if '4' in BIN_TABLE.index else np.nan}
HEX_TABLE['ideal min spacing [km] (a = 12.1)'] = HEX_TABLE['min spacing [a]'] * NN_FULL.mean()
HEX_TABLE['real grid, Fibonacci bins: NN min / p05 [km]'] = [
    f"{BIN_TABLE.loc[k, 'NN min [km]']:.1f} / {BIN_TABLE.loc[k, 'NN p05 [km]']:.1f}" if k in BIN_TABLE.index else '-'
    for k in ['2', '2', '2', '3', '4']]
print(HEX_TABLE.to_string(index=False, float_format=lambda v: f'{v:.3f}'))

Red lines: nearest-neighbour pairs at the full spacing $a$ that survive keep-1-in-2. Black lines (1/3, 1/4): the
nearest-neighbour links of the kept points, at the larger spacing. On the real grid, 1-in-3 and 1-in-4
fall short of the ideal ($\sqrt3\,a$, $2a$) because a coarse lattice that is not aligned with the fine one leaves
boundary pairs. And no single regular sub-pattern can be used on the whole Fibonacci grid, because its local
neighbour directions change with latitude (§1.4): any such pattern would need seams.

**Takeaway:** keeping half the obs doubles the area per obs, so the *average* spacing grows by √2, but on a hexagonal
grid it cannot increase the distance to the *nearest* neighbour. That is a property of the lattice, not of the
selection method.

### 4.1 Contrast: a regular square grid (e.g. EASE2 M09)

On a regular grid each point has a row and column index $(i, j)$, and its nearest neighbours are always $(i\pm1, j)$ and
$(i, j\pm1)$: **4 neighbours along 2 directions**. Index arithmetic then controls exactly which neighbours are dropped:

- **every other column** (`i % 2 == 0`; also what "every other obs" gives on a flattened index with an even number of
  columns, e.g. M09's 3856): the north–south neighbours remain, so the minimum spacing does not change;
- **checkerboard** (`(i + j) % 2 == 0`): all 4 neighbours are dropped, so keeping half the obs **does** increase the
  minimum spacing, by √2 (unlike any keep-half pattern on the hexagonal grid above);
- **every other row and column** (`i % 2 == 0 and j % 2 == 0`): 1/4 kept, spacing doubled.

The Fibonacci grid has neither property: it is locally hexagonal (6 neighbours along 3 directions), and it has no
$(i, j)$ at all, only a single index along a spiral, so even a checkerboard cannot be written down.

In [ ]:
# ideal square lattice, spacing a = 1 (e.g. one M09 cell, ~9 km)
ii, jj = np.meshgrid(np.arange(-12, 13), np.arange(-12, 13))
ii, jj = ii.ravel(), jj.ravel()
SQ = np.c_[ii, jj].astype(float)
SQ_SUBS = {'every other column: i % 2 == 0 (1/2)': ii % 2 == 0,
           'checkerboard: (i + j) % 2 == 0 (1/2)': (ii + jj) % 2 == 0,
           'every other row and column (1/4)': (ii % 2 == 0) & (jj % 2 == 0)}

fig, axs = plt.subplots(1, 3, figsize=(16, 5.4), constrained_layout=True)
rows = []
for ax, (name, m) in zip(axs, SQ_SUBS.items()):
    s = min_spacing_2d(SQ[m])
    rows.append({'pattern': name, 'kept fraction': m.mean(), 'min spacing [a]': s})
    ax.plot(SQ[~m, 0], SQ[~m, 1], 'o', color=C_ALL, ms=4)
    ax.plot(SQ[m, 0], SQ[m, 1], 'o', color=C_KEEP, ms=7)
    pr = cKDTree(SQ[m]).query_pairs(s + 1e-9, output_type='ndarray')
    draw_pairs(ax, SQ[m][:, 1], SQ[m][:, 0], pr, colors=C_BAD if s < 1.01 else 'k', linewidths=1.2)
    ax.set_xlim(-4, 4); ax.set_ylim(-3.5, 3.5); ax.set_aspect('equal')
    ax.set_xlabel('column i'); ax.set_ylabel('row j')
    ax.set_title(f'{name}\nmin spacing {s:.3f} a', fontsize=11)
fig.savefig(FIG_DIR / 'square_sublattices.png', dpi=150)
plt.show()

A_M09 = 9.008                                   # EASE2 M09 cell size [km] (true scale at +-30 deg)
sq = pd.DataFrame(rows)
sq['M09 min spacing [km]'] = sq['min spacing [a]'] * A_M09
print('square grid (red = nearest-neighbour links at the full spacing that survive; black = larger spacing):')
print(sq.to_string(index=False, float_format=lambda v: f'{v:.3f}'))
print('\nhexagonal grid (from the figure above), for comparison:')
print(HEX_TABLE[['pattern', 'kept fraction', 'min spacing [a]']].to_string(index=False, float_format=lambda v: f'{v:.3f}'))

# EASE2 global is cylindrical equal-area, true scale at 30 deg: cells are not square away from +-30 deg
rows = []
for lat in [0, 30, 45, 60, 70]:
    ew = A_M09 * np.cos(np.radians(lat)) / np.cos(np.radians(30))
    ns = A_M09 * np.cos(np.radians(30)) / np.cos(np.radians(lat))
    rows.append({'lat': lat, 'M09 cell E-W [km]': ew, 'M09 cell N-S [km]': ns,
                 'checkerboard min spacing [km]': min(np.hypot(ew, ns), 2 * ew, 2 * ns)})
print('\nEASE2 M09 cell shape vs latitude (index thinning inherits this anisotropy):')
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f'{v:.1f}'))

So on a regular grid such as M09, index thinning is a reasonable cheap option: the checkerboard keeps half the obs
with √2 × the spacing, and every other row and column keeps a quarter with 2 × the spacing. Two caveats remain: (1)
EASE2 cells are only square near ±30°, so index thinning gives direction-dependent spacing poleward (table above);
(2) an index rule is a **static mask**: if an allowed obs fails QC its area gets nothing, even when a valid neighbour
exists, whereas thinning after QC (binning or the greedy method) uses the next available obs.

**Takeaway:** "every other obs" works on a square grid (as a checkerboard) because its nearest neighbours lie along just
two directions. On the hexagonal, spiral-indexed Fibonacci grid no index pattern can do that, which is why we moved to
binning and then to a minimum distance.

## 5. A reduced lat/lon grid bins just as well

`thin_reduced_latlon` with rows $h$ = 12.5·√(`THIN_RATIO`) km tall and ~$h$-wide columns, against Fibonacci binning with
the same `THIN_RATIO`, on all points and on the 70%-valid set of §3.3.

In [ ]:
h_km = 12.5 * np.sqrt(THIN_RATIO)
rows = []
for valid_name, m in [('100%', np.ones(I_FULL.size, bool)), ('70%', VALID)]:
    la, lo = LAT_GD[m], LON_FULL[m]
    for meth, fn in [('Fibonacci', lambda: thin_fib_bins(la, lo, int(round(N_FINE / THIN_RATIO)))),
                     ('reduced lat/lon', lambda: thin_reduced_latlon(la, lo, h_km))]:
        k = fn()
        d = nn_spacing_km(la[k], lo[k])
        band = spacing_by_lat_band(la[k], lo[k], d=d)
        rows.append({'valid': valid_name, 'method': meth, 'kept': f'{k.size / la.size:.1%}', 'NN mean': d.mean(),
                     'p05': np.percentile(d, 5), 'band range': f'{min(band.values()):.1f}-{max(band.values()):.1f}'})
LL_TABLE = pd.DataFrame(rows)
print(f'thin_ratio = {THIN_RATIO:g}, reduced lat/lon h = {h_km:.1f} km')
print(LL_TABLE.to_string(index=False, float_format=fmt))

**Takeaway:** a coarse Fibonacci lattice is just a set of roughly equal-area bins. Any equal-area binning behaves the
same, including its boundary pairs. The Fibonacci lattice brings no special benefit.

## 6. Greedy minimum-distance thinning (implemented: `thin_dist_km`)

**Rule:** visit obs in order; keep an obs unless it is closer than $r$ (= `thin_dist_km`) to an obs kept earlier.
By construction, no two kept obs are closer than $r$.

### 6.2 One raw file (no QC), file order

In [ ]:
with nc.Dataset(TEST_FILE) as d:
    d.set_auto_maskandscale(False)
    RAW_LAT = d['latitude'][:].astype(np.float32) * np.float32(1e-6)    # single precision, as in the reader
    RAW_LON = d['longitude'][:].astype(np.float32) * np.float32(1e-6)
rows, RAW_KEEP = [], {}
for r in THIN_DISTS_COMPARE:
    k = thin_min_dist(*as64(RAW_LAT, RAW_LON), r)
    RAW_KEEP[r] = k
    rows.append({'thin_dist_km': r, 'kept': k.sum(), 'kept %': 100 * k.mean(),
                 'min separation [km]': min_sep_km(*as64(RAW_LAT[k], RAW_LON[k]))})
RAW_TABLE = pd.DataFrame(rows)
print(f'{Path(TEST_FILE).name}: {RAW_LAT.size:,} obs (no QC)')
print(RAW_TABLE.to_string(index=False, float_format=lambda v: f'{v:.2f}'))

### 6.3 QC'd obs of an assimilation window

In [ ]:
WIN = read_hsaf_window(SAT, CYCLE)
lat_w, lon_w = as64(WIN['lat'], WIN['lon'])
print(WIN['label'])
print(WIN['table'].to_string(index=False))

# original proposal: per file, coarse Fibonacci bins (as in the ca872c7 reader)
def thin_ratio_per_file(W, ratio):
    keep = np.zeros(W['lat'].size, bool)
    for f in np.unique(W['file']):
        idx = np.flatnonzero(W['file'] == f)
        keep[idx[thin_fib_bins(*as64(W['lat'][idx], W['lon'][idx]), n_thin(ratio))]] = True
    return keep

KEEP = {'no thinning': np.ones(lat_w.size, bool)}
for r in THIN_RATIOS_COMPARE:
    KEEP[f'thin_ratio={r:g}'] = thin_ratio_per_file(WIN, r)
for r in THIN_DISTS_COMPARE:
    KEEP[f'thin_dist_km={r:g}'] = thin_min_dist(lat_w, lon_w, r)

rows, WIN_NN = {}, {}
for name, k in KEEP.items():
    d = nn_spacing_km(lat_w[k], lon_w[k])
    WIN_NN[name] = d
    band = spacing_by_lat_band(lat_w[k], lon_w[k], d=d)
    rows[name] = {'kept': k.sum(), 'kept % of QC': 100 * k.mean(), 'NN min': d.min(), 'NN p05': np.percentile(d, 5),
                  'NN median': np.median(d), 'band range of mean NN': f'{min(band.values()):.1f}-{max(band.values()):.1f}'}
WIN_TABLE = pd.DataFrame(rows).T
print(f'\nobs in files {WIN["table"]["obs in file"].sum():,} -> passing QC {lat_w.size:,} -> kept (NN distances in km):')
print(WIN_TABLE.to_string(float_format=fmt))

### 6.4 Figures: nearest-neighbour distances, and the obs layout for each setting

**Histogram:** greedy thinning has a hard floor at $r$; binning has a tail down to ~12 km.

**Layout:** the same patch inside a swath for every setting. Top row: the original proposal (`thin_ratio`), with red
lines for kept pairs closer than 13 km. Bottom row: the implemented method (`thin_dist_km`), with a circle of radius
$r/2$ around each kept obs: the circles never overlap.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
names = [f'thin_ratio={r:g}' for r in THIN_RATIOS_COMPARE if r in (THIN_RATIO, 4.0)] + [f'thin_dist_km={THIN_DIST_KM:g}']
for name in dict.fromkeys(names):
    ax.hist(WIN_NN[name], bins=np.arange(10, 40, 0.25), histtype='step', lw=2, density=True, label=name)
ax.axvline(THIN_DIST_KM, color='k', ls=':', label=f'r = {THIN_DIST_KM:g} km')
ax.set_xlabel('nearest-neighbour distance of kept obs [km]'); ax.set_ylabel('density'); ax.legend()
ax.set_title(WIN['label'])
fig.savefig(FIG_DIR / 'nn_hist_bins_vs_greedy.png', dpi=150)
plt.show()

ncol = 1 + max(len(THIN_RATIOS_COMPARE), len(THIN_DISTS_COMPARE))
fig, axs = plt.subplots(2, ncol, figsize=(5.0 * ncol, 10), constrained_layout=True)
bp = patch_mask(lat_w, lon_w, PATCH_SWATH)
t = np.linspace(0, 2 * np.pi, 60)
for row, (prefix, vals) in enumerate([('thin_ratio', THIN_RATIOS_COMPARE), ('thin_dist_km', THIN_DISTS_COMPARE)]):
    ax = axs[row, 0]
    ax.plot(lon_w[bp], lat_w[bp], 'o', color=C_KEEP, ms=3)
    set_patch(ax, PATCH_SWATH)
    ax.set_title(f'no thinning: {bp.sum()} QC\'d obs')
    for c in range(1, ncol):
        ax = axs[row, c]
        if c > len(vals):
            ax.axis('off'); continue
        v = vals[c - 1]
        k = KEEP[f'{prefix}={v:g}'] & bp
        ax.plot(lon_w[bp], lat_w[bp], 'o', color=C_ALL, ms=2.5)
        la, lo = lat_w[k], lon_w[k]
        if prefix == 'thin_ratio':
            pr = close_pairs(la, lo, 13.0)
            draw_pairs(ax, la, lo, pr, colors=C_BAD, linewidths=2, zorder=3)
            extra = f'{len(pr)} pairs < 13 km'
        else:
            rr = np.degrees(v / 2 / R_EARTH_KM)
            for x, y in zip(lo, la):
                ax.plot(x + rr * np.cos(t) / np.cos(np.radians(y)), y + rr * np.sin(t), '-', color=C_KEEP, lw=0.6)
            extra = f'min sep {min_sep_km(la, lo):.1f} km'
        ax.plot(lo, la, 'o', color=C_KEEP, ms=4, zorder=4)
        set_patch(ax, PATCH_SWATH)
        ax.set_title(f'{prefix} = {v:g}: {k.sum()} kept, {extra}')
fig.suptitle(f'{WIN["label"]}, patch {PATCH_SWATH}', fontsize=14)
fig.savefig(FIG_DIR / 'layout_all_settings.png', dpi=150)
plt.show()

**Takeaway:** greedy thinning gives exactly the property we want, a hard minimum distance between assimilated obs, at
negligible cost. GEOSldas implements it as `obs_param_nml(56:58)%thin_dist_km` (subroutine `thin_min_dist` in `clsm_ensupd_read_obs.F90`).

## 7. Caveats and choices

### 7.1 Order dependence

The same window and `THIN_DIST_KM`, visiting the obs in file order (GEOSldas), in a seeded random order, and by
descending SSM sensitivity.

In [ ]:
orders = {'file order (GEOSldas)': np.arange(lat_w.size),
          'random (seed 0)': np.random.default_rng(0).permutation(lat_w.size),
          'descending sensitivity': np.lexsort((np.arange(lat_w.size), -WIN['sens']))}
SEL = {}
for name, o in orders.items():
    k = np.zeros(lat_w.size, bool)
    k[o] = thin_min_dist(lat_w[o], lon_w[o], THIN_DIST_KM)
    SEL[name] = k
    print(f'{name:24s} kept {k.sum():7,}, min separation {min_sep_km(lat_w[k], lon_w[k]):.2f} km')
names = list(SEL)
ov = pd.DataFrame([[(SEL[a] & SEL[b]).sum() / (0.5 * (SEL[a].sum() + SEL[b].sum())) for b in names] for a in names],
                  index=names, columns=names)
print('\nfraction of kept obs common to two orders (|A∩B| / mean(|A|,|B|)):')
print(ov.to_string(float_format=lambda v: f'{v:.2f}'))

### 7.2 Scope: per file vs per window

Thin each hourly file separately, then count kept pairs closer than $r$ **across** files of the window. These are
the same grid points seen again by the next orbit ~100 min later, where swaths overlap. That is why GEOSldas collects
the whole window before thinning (the original `thin_ratio` reader thinned per file).

In [ ]:
keep_pf = np.zeros(lat_w.size, bool)
for f in np.unique(WIN['file']):
    idx = np.flatnonzero(WIN['file'] == f)
    keep_pf[idx] = thin_min_dist(lat_w[idx], lon_w[idx], THIN_DIST_KM)
kk = np.flatnonzero(keep_pf)
pr = close_pairs(lat_w[kk], lon_w[kk], THIN_DIST_KM)
cross = WIN['file'][kk[pr[:, 0]]] != WIN['file'][kk[pr[:, 1]]]
print(f'thin_dist_km = {THIN_DIST_KM:g}, per file: kept {keep_pf.sum():,}; kept pairs closer than r: {len(pr):,} '
      f'({cross.sum():,} across files); per window (GEOSldas): kept {KEEP[f"thin_dist_km={THIN_DIST_KM:g}"].sum():,}, 0 pairs')
if cross.any():
    lat_c = lat_w[kk[pr[cross, 0]]]
    print('|lat| of the cross-file pairs: ', np.percentile(np.abs(lat_c), [5, 50, 95]).round(1), '(5/50/95th pct)')

### 7.3 Across satellites

Metop-A/B/C are separate species, so GEOSldas thins each on its own. For a window where Metop-B and Metop-C both have
data, count kept Metop-C obs closer than $r$ to a kept Metop-B obs.

In [ ]:
WB, WC = read_hsaf_window('b', CYCLE_BC), read_hsaf_window('c', CYCLE_BC)
if WB['lat'].size and WC['lat'].size:
    lb, ob = as64(WB['lat'], WB['lon']); lc, oc = as64(WC['lat'], WC['lon'])
    kb, kc = thin_min_dist(lb, ob, THIN_DIST_KM), thin_min_dist(lc, oc, THIN_DIST_KM)
    d, _ = cKDTree(to_xyz(lb[kb], ob[kb])).query(to_xyz(lc[kc], oc[kc]))
    close = chord_to_km(d) < THIN_DIST_KM
    print(f'{WB["label"]}: kept {kb.sum():,};  {WC["label"]}: kept {kc.sum():,}')
    print(f'kept Metop-C obs closer than {THIN_DIST_KM:g} km to a kept Metop-B obs: {close.sum():,} ({close.mean():.0%})')
else:
    print('no Metop-B/Metop-C overlap: one of the windows is empty; choose another CYCLE_BC')

### 7.4 Thresholds on lattice distances are sensitive to rounding

The reader passes lat/lon in single precision. For most `thin_dist_km` that does not matter, but the 12.5 km grid has
many pairs at almost exactly a lattice distance (e.g. ~25 km ≈ 2 × spacing). A threshold right there decides those pairs
on the last bits of precision, and the greedy pass cascades the difference. Below: pair distances in the raw test file,
and the kept count for three ways of forming the coordinates: as the reader does (`real(lat_raw)*real(1e-6)`, single
precision), scaled in double precision and then rounded to float32, and float64. The obs are the same in all three;
only the last bits of their coordinates differ.

In [ ]:
pr = close_pairs(*as64(RAW_LAT, RAW_LON), 30.0)
p = to_xyz(*as64(RAW_LAT, RAW_LON))
dpair = chord_to_km(np.sqrt(((p[pr[:, 0]] - p[pr[:, 1]]) ** 2).sum(1)))
fig, ax = plt.subplots(figsize=(11, 4), constrained_layout=True)
ax.hist(dpair, bins=np.arange(10, 30, 0.05), color='0.4')
for r in THIN_DISTS_COMPARE:
    ax.axvline(r, color=C_BAD, ls='--', lw=1)
ax.set_xlabel('distance between obs pairs [km]'); ax.set_ylabel('pairs'); ax.set_yscale('log')
ax.set_title(f'pair distances in {Path(TEST_FILE).name[:60]}...; dashed = thin_dist_km values compared')
fig.savefig(FIG_DIR / 'pair_distances.png', dpi=150)
plt.show()

rows, RAW_KEEP_ALT = [], {}
with nc.Dataset(TEST_FILE) as d:
    lat64, lon64 = np.asarray(d['latitude'][:], float), np.asarray(d['longitude'][:], float)   # int * 1e-6 in double
for r in THIN_DISTS_COMPARE:
    k_alt = thin_min_dist(*as64(lat64.astype(np.float32), lon64.astype(np.float32)), r)
    RAW_KEEP_ALT[r] = k_alt
    rows.append({'thin_dist_km': r, 'pairs within ±0.01 km of r': int((np.abs(dpair - r) < 0.01).sum()),
                 'kept: real(raw)*real(1e-6) (reader)': RAW_KEEP[r].sum(),
                 'kept: double-scaled, then float32': k_alt.sum(),
                 'kept: float64': thin_min_dist(lat64, lon64, r).sum()})
print(pd.DataFrame(rows).to_string(index=False))

**Takeaway:** what is guaranteed is a minimum distance **per species, per assimilation window, in file order**. It is
not enforced between satellites, and which obs are kept depends on the visiting order. Avoid `thin_dist_km` values that
sit exactly on a lattice distance (see 7.4) if bit-for-bit reproducibility across compilers matters.

## 8. What reaches the CF0360 tiles

The reader (`5e220ff`) first assigns every QC'd obs of the window to a tile, then thins **only the obs that have a
tile**, and finally **averages all kept obs that fall in the same tile** into one tile obs (mean SSM, mean lat/lon).
(`a499c55` thinned first and then assigned tiles; the order table below compares the two.) Tile assignment follows the rule checked in `cf0360_grid_and_tiles.ipynb` §6: the
obs's cell on the 0.25° × 1/6° "pert grid", then the land tile listed in that cell with the nearest centre of mass
(|Δlon| + |Δlat|). The max-distance check is not reproduced (it only drops obs).

The tile obs positions are what the EnKF sees, so the separation that matters in the end is between tile obs.

In [ ]:
TC = read_tilecoord(str(TILECOORD))
tlat, tlon = TC['com_lat'].astype(float), TC['com_lon'].astype(float)
PERT_NLON, PERT_NLAT = 4 * 360, 3 * 360                      # DE1440xPE1080 for C360

def pert_key(lat, lon):
    return (np.floor((lat + 90) / (180 / PERT_NLAT)).astype(np.int64) * PERT_NLON
            + np.floor((lon + 180) / (360 / PERT_NLON)).astype(np.int64))

tkey = pert_key(tlat, tlon)
torder = np.argsort(tkey, kind='stable')
tkey_s = tkey[torder]

def tile_of_obs(lat, lon):
    key = pert_key(lat, lon)
    lo_, hi_ = np.searchsorted(tkey_s, key), np.searchsorted(tkey_s, key, side='right')
    out = np.full(lat.size, -1)
    for n in np.flatnonzero(hi_ > lo_):
        cand = torder[lo_[n]:hi_[n]]
        out[n] = cand[np.argmin(np.abs(tlon[cand] - lon[n]) + np.abs(tlat[cand] - lat[n]))] + 1
    return out

TILE_W = tile_of_obs(lat_w, lon_w)            # assignment does not depend on thinning: do it once

# as the reader (5e220ff): thin_dist_km thins only the obs that were assigned to a tile.
# (thin_ratio, the never-implemented proposal, stays as it was: per file, before tile matching.)
HAS_TILE = TILE_W > 0
KEEP_GEOS = {}
for name, k in KEEP.items():
    if name.startswith('thin_dist_km='):
        kk = np.zeros(lat_w.size, bool)
        kk[HAS_TILE] = thin_min_dist(lat_w[HAS_TILE], lon_w[HAS_TILE], float(name.split('=')[1]))
        KEEP_GEOS[name] = kk
    else:
        KEEP_GEOS[name] = k

def tile_obs(keep):
    t = TILE_W[keep & (TILE_W > 0)]
    la, lo = lat_w[keep & (TILE_W > 0)], lon_w[keep & (TILE_W > 0)]
    u, inv, cnt = np.unique(t, return_inverse=True, return_counts=True)
    return u, cnt, np.bincount(inv, la) / cnt, np.bincount(inv, lo) / cnt     # tiles, obs per tile, mean lat/lon

rows, TILE_OBS = {}, {}
for name, k in KEEP_GEOS.items():
    u, cnt, mla, mlo = tile_obs(k)
    TILE_OBS[name] = (u, mla, mlo)
    d = nn_spacing_km(mla, mlo)
    rows[name] = {'obs to tiles': cnt.sum(), 'tiles with obs': u.size, 'obs per tile obs': cnt.mean(),
                  'tile obs NN min [km]': d.min(), 'tile obs NN p05 [km]': np.percentile(d, 5),
                  'tile obs NN median [km]': np.median(d)}
TILE_TABLE = pd.DataFrame(rows).T
print(f'{TC["N_tile"]:,} CF0360 land tiles; {WIN["label"]}')
print(TILE_TABLE.to_string(float_format=lambda v: f'{v:.2f}'))

# order of thinning and tile matching: a499c55 (thin all QC'd obs, then match) vs 5e220ff (match, then thin)
rows = {}
for name in [n for n in KEEP if n.startswith('thin_dist_km=')]:
    old, new = KEEP[name] & HAS_TILE, KEEP_GEOS[name]
    rows[name] = {'QC\'d obs': lat_w.size, 'without tile': (~HAS_TILE).sum(),
                  'kept with tile, thin first (a499c55)': old.sum(), 'kept with tile, tile first (5e220ff)': new.sum(),
                  'gained': (new & ~old).sum(), 'lost': (old & ~new).sum()}
ORDER_TABLE = pd.DataFrame(rows).T
print('\nthinning order (obs that reach a tile):')
print(ORDER_TABLE.to_string())

In a patch: tile centres of mass (small grey dots), each kept obs joined to the tile it is assigned to, and the tile obs
(mean position, orange) of each tile that gets one. Without thinning, a tile can average two neighbouring obs; with
`thin_dist_km` every tile obs is one obs, and tile obs keep the minimum separation (up to the averaging).

In [ ]:
PATCH_T = (PATCH_SWATH[0], PATCH_SWATH[0] + 1.0, PATCH_SWATH[2], PATCH_SWATH[2] + 1.0)
show = ['no thinning', f'thin_ratio={THIN_RATIO:g}', f'thin_dist_km={THIN_DIST_KM:g}']
fig, axs = plt.subplots(1, len(show), figsize=(6.5 * len(show), 6.8), constrained_layout=True)
bt = patch_mask(tlat, tlon, (PATCH_T[0] - 0.2, PATCH_T[1] + 0.2, PATCH_T[2] - 0.2, PATCH_T[3] + 0.2))
for ax, name in zip(axs, show):
    k = KEEP_GEOS[name] & patch_mask(lat_w, lon_w, PATCH_T) & (TILE_W > 0)
    ax.plot(tlon[bt], tlat[bt], '.', color='0.6', ms=3, label='tile centre of mass')
    seg = np.stack([np.c_[lon_w[k], lat_w[k]], np.c_[tlon[TILE_W[k] - 1], tlat[TILE_W[k] - 1]]], axis=1)
    ax.add_collection(LineCollection(seg, colors='0.3', linewidths=0.7))
    ax.plot(lon_w[k], lat_w[k], 'o', color=C_KEEP, ms=4, label='kept obs')
    u, mla, mlo = TILE_OBS[name]
    mt = patch_mask(mla, mlo, PATCH_T)
    ax.plot(mlo[mt], mla[mt], 's', mfc='none', mec='tab:orange', ms=7, mew=1.3, label='tile obs (mean position)')
    set_patch(ax, PATCH_T)
    n_multi = np.isin(u, pd.Series(TILE_W[k]).value_counts().index[pd.Series(TILE_W[k]).value_counts() > 1]).sum()
    ax.set_title(f'{name}: {k.sum()} obs -> {mt.sum()} tile obs ({n_multi} tiles average > 1 obs)')
    ax.legend(loc='lower left', fontsize=8)
fig.savefig(FIG_DIR / 'obs_to_tiles.png', dpi=150)
plt.show()

**Takeaway:** CF0360 tiles are small, so almost every kept obs becomes its own tile obs and the number of tile obs drops
almost in proportion to the thinning. With `thin_dist_km` the tile obs keep (nearly) the minimum separation;
with `thin_ratio` or no thinning, tile obs ~12 km apart remain.

**Order of tile matching and thinning** (order table). In this global window only ~1% of the QC'd obs have no tile,
so matching tiles first (`5e220ff`) keeps only slightly more obs than thinning first (`a499c55`): +149 at 20 km
(+0.6%). But greedy thinning cascades: removing a few candidates early in the file order changes which of their
neighbours are kept, and those changes propagate. So ~15% of the kept obs differ between the two orders (3,631 gained,
3,482 lost at 20 km). Both selections keep the minimum separation. The gain is larger where many obs have no tile:
along the edges of regional domains, and along coasts when the reader's max-distance check also drops obs (it is
not reproduced here).

## 9. Summary

In [ ]:
SUMMARY = pd.DataFrame([
    {'method': 'gpi parity (i % k)', 'uniform across latitude': 'no', 'controls the count': 'yes',
     'guarantees a minimum distance': 'no', 'complexity': 'trivial'},
    {'method': 'Fibonacci binning (thin_ratio, original proposal)', 'uniform across latitude': 'yes',
     'controls the count': 'yes', 'guarantees a minimum distance': 'no (pairs ~12 km remain)',
     'complexity': 'coarse lattice + nearest-point search'},
    {'method': 'reduced lat/lon binning', 'uniform across latitude': 'yes', 'controls the count': 'yes',
     'guarantees a minimum distance': 'no (same boundary pairs)', 'complexity': 'simple arithmetic'},
    {'method': 'greedy minimum distance (thin_dist_km, implemented)', 'uniform across latitude': 'yes',
     'controls the count': 'indirectly (via r)', 'guarantees a minimum distance': 'yes, >= r',
     'complexity': 'one pass + bucket grid'},
]).set_index('method')
print(SUMMARY.to_string())
print('\nthis run, ' + WIN['label'] + ':')
print(WIN_TABLE[['kept', 'kept % of QC', 'NN min', 'NN p05']].to_string(float_format=fmt))

The requirement is a minimum separation between assimilated obs, to limit footprint overlap and correlated errors.
Binning (on a Fibonacci or any other equal-area grid) controls the count but cannot guarantee separation, and at
1-in-2 it hardly changes the nearest-neighbour distance (§3–5). Greedy minimum-distance thinning guarantees
it, is cheap, and is what GEOSldas now does (`thin_dist_km`, per species and per window, among the obs assigned to a tile, in file order).

## Acceptance checks

The brief's expected values are for `THIN_RATIO = 2` and `THIN_DISTS_COMPARE` containing 15, 20 and 25 km; checks whose
inputs are not in this run are skipped.

In [ ]:
CHECKS = []
def check(name, ok, detail=''):
    CHECKS.append({'check': name, 'result': 'PASS' if ok else 'FAIL', 'detail': detail})

# 1. location_id mapping + geodetic lat
check('1. location_id mapping + geodetic lat match the file to < 1e-3 deg',
      MAP_CHECK.loc['(c) correct mapping, geodetic lat'].max() < 1e-3,
      f"max |dlat| {MAP_CHECK.iloc[2, 0]:.1e}, |dlon| {MAP_CHECK.iloc[2, 1]:.1e}")

# 2. i % 4 band table
exp4 = np.array([12.2, 12.6, 13.8, 17.0, 23.3, 20.8])
got4 = IDX_TABLE.loc['i % 4 == 0'].to_numpy()
check('2. i % 4 band table within 0.2 km of the brief', np.all(np.abs(got4 - exp4) <= 0.2), f'{np.round(got4, 1)}')

# 3. binning table
exp3 = {'1 (all)': (100, 12.4, 12.1, 11.8), '2': (50.0, 17.6, 12.8, 11.8), '3': (33.3, 21.5, 15.3, 12.2),
        '4': (25.0, 24.9, 20.1, 12.4)}
ok3, det3 = True, []
for k, e in exp3.items():
    if k not in BIN_TABLE.index:
        continue
    g_ = BIN_TABLE.loc[k, ['kept', 'sqrt(area/ob) [km]', 'NN mean [km]', 'NN p05 [km]']].to_numpy(float)
    ok3 &= bool(np.all(np.abs(g_ - np.array(e)) <= 0.2))
    det3.append(f'{k}: {np.round(g_, 1).tolist()}')
check('3. binning table within 0.2 km / 0.2 pp of the brief', ok3, '; '.join(det3))

# 4. raw-file kept counts
exp6 = {15.0: 92682, 20.0: 59556, 25.0: 44668}
got6 = {r: int(RAW_KEEP_ALT[r].sum()) for r in exp6 if r in RAW_KEEP_ALT}
got6_reader = {r: int(RAW_KEEP[r].sum()) for r in exp6 if r in RAW_KEEP}
check('4. raw-file greedy kept counts match the brief exactly (brief\'s coordinate path)',
      all(got6[r] == exp6[r] for r in got6), f'{got6}; reader path: {got6_reader}')

# 5. minimum separation of every greedy result
seps = [min_sep_km(*as64(RAW_LAT[k], RAW_LON[k])) / r for r, k in RAW_KEEP.items()]
seps += [WIN_TABLE.loc[f'thin_dist_km={r:g}', 'NN min'] / r for r in THIN_DISTS_COMPARE]
seps += [min_sep_km(lat_w[k], lon_w[k]) / THIN_DIST_KM for k in SEL.values()]
check('5. every greedy result has min separation >= r (1 - 1e-9)', min(seps) >= 1 - 1e-9, f'min ratio {min(seps):.9f}')

# 6. determinism, and fast greedy == literal reference
k1 = thin_min_dist(lat_w, lon_w, THIN_DIST_KM); k2 = thin_min_dist(lat_w, lon_w, THIN_DIST_KM)
kb1 = thin_fib_bins(lat_w, lon_w, n_thin(THIN_RATIO)); kb2 = thin_fib_bins(lat_w, lon_w, n_thin(THIN_RATIO))
# the literal pure-Python reference is slow (cost ~ obs x neighbours within r); compare on a subset of the window
N_REF = min(20000, lat_w.size)
kref = thin_min_dist_reference(lat_w[:N_REF], lon_w[:N_REF], THIN_DIST_KM)
kfast = thin_min_dist(lat_w[:N_REF], lon_w[:N_REF], THIN_DIST_KM)
check('6. thinning is deterministic, and thin_min_dist == thin_min_dist_reference',
      np.array_equal(k1, k2) and np.array_equal(kb1, kb2) and np.array_equal(kfast, kref),
      f'reference compared on the first {N_REF:,} obs of the window')

# 7. runtime
elapsed = time.time() - T_START
check('7. notebook runtime < ~5 min', elapsed < 300, f'{elapsed:.0f} s')

print(pd.DataFrame(CHECKS).to_string(index=False))

**Note on check 4.** The brief's counts (92,682 / 59,556 / 44,668) are reproduced exactly when lat/lon are scaled in
double precision and then rounded to float32, which is evidently how they were produced. The reader instead forms
`real(lat_raw)*real(1e-6)` in single precision, which gives slightly different coordinates and so 92,676 / 59,563 /
44,390 kept obs. The compiled Fortran gives exactly these reader-path numbers (standalone test). All variants satisfy the minimum
separation. The counts differ only because many pairs sit almost exactly at a lattice distance (§7.4).